# 36 · Scale-Weighted Residual Model

This notebook upgrades Notebook 35 by making the residual correction explicitly **scale-weighted**.

Notebook 35 suggested:
- the unified core captures the shared boundary
- residual corrections improve separation
- residual magnitude grows with window size
- but residual usefulness is not monotone with window size

That motivates the model

\[
f(x; k) = f_{\mathrm{core}}(x) + \lambda(k) \, r(x)
\]

where:
- \(f_{\mathrm{core}}\) is the unified boundary
- \(r(x)\) is the learned residual correction
- \(\lambda(k)\) is a scale-dependent residual weight

## Model families

We compare:
- **Unified core**
- **Specialist boundary**
- **Residual model**
- **Scale-weighted residual model**

## Main question

For each window size \(k\), what residual weight \(\lambda\) gives the best phase-gap performance?

## Outputs

- phase gap vs residual weight
- best residual weight by scale
- weighted-residual vs unweighted-residual vs unified vs specialist bars
- weighted-residual gain heatmaps
- residual-weight summaries

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 2200
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)
        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        all_gaps.extend(np.diff(bulk_vals).tolist())
    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Window statistics and feature builder

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def ratio_std(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.std(r))

def build_windows_and_features(gaps, feature_family="minimal", k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])

    stats = {
        "entropy": np.array([window_entropy(w) for w in windows_n]),
        "unevenness": np.array([unevenness(w) for w in windows_n]),
        "ratio_mean": np.array([ratio_mean(w) for w in windows_n]),
        "symmetry": np.array([reversal_symmetry_score(w) for w in windows_n]),
    }

    if feature_family == "minimal":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["ratio_mean"][i]]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "full":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["symmetry"][i], stats["ratio_mean"][i], ratio_std(windows_n[i])]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "raw_window":
        X = windows_n.copy()
    else:
        raise ValueError(f"Unknown feature family: {feature_family}")

    return windows_n, stats, X

def apply_condition_mask(stats, condition_name):
    if condition_name == "low_entropy":
        thr = np.median(stats["entropy"])
        return stats["entropy"] <= thr
    if condition_name == "high_entropy":
        thr = np.median(stats["entropy"])
        return stats["entropy"] > thr
    if condition_name == "low_unevenness":
        thr = np.median(stats["unevenness"])
        return stats["unevenness"] <= thr
    if condition_name == "high_unevenness":
        thr = np.median(stats["unevenness"])
        return stats["unevenness"] > thr
    raise ValueError(condition_name)

def make_contextual_features(X):
    prev_X = np.vstack([X[0], X[:-1]])
    next_X = np.vstack([X[1:], X[-1]])
    delta_prev = X - prev_X
    delta_next = next_X - X
    curv = next_X - 2 * X + prev_X
    return np.hstack([X, delta_prev, delta_next, curv])

## Core helpers

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

def fit_logistic_regression(X, y, lr=0.1, n_steps=2500, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])
    for _ in range(n_steps):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(X)
        grad[1:] += reg * w[1:]
        w -= lr * grad
    return w

def decision_function_logistic(X, w):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ w

def predict_proba_logistic(X, w):
    return sigmoid(decision_function_logistic(X, w))

def choose_prototypes(X, n_proto=20):
    idx = np.linspace(0, len(X)-1, n_proto).astype(int)
    return X[idx]

def estimate_rbf_gamma(X):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)
    pos = D[D > 0]
    med = np.median(pos) if len(pos) else 1.0
    if med <= 0:
        med = 1.0
    return 1.0 / (2.0 * med * med)

def rbf_features(X, prototypes, gamma):
    D2 = np.sum((X[:, None, :] - prototypes[None, :, :]) ** 2, axis=2)
    return np.exp(-gamma * D2)

def overlap_coefficient_from_hist(a, b, bins=30):
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    if hi <= lo:
        return 1.0
    hist_a, edges = np.histogram(a, bins=bins, range=(lo, hi), density=True)
    hist_b, _ = np.histogram(b, bins=bins, range=(lo, hi), density=True)
    dx = edges[1] - edges[0]
    return float(np.sum(np.minimum(hist_a, hist_b)) * dx)

def evaluate_scores(y_true, scores, probs):
    preds = (probs >= 0.5).astype(int)
    acc = float(np.mean(preds == y_true))
    s0 = scores[y_true == 0]
    s1 = scores[y_true == 1]
    overlap = overlap_coefficient_from_hist(s0, s1, bins=30)
    return {"accuracy": acc, "overlap": overlap}

## Unified, specialist, residual, and scale-weighted residual

In [ ]:
def fit_rbf_boundary(X0_train, X1_train):
    m = min(len(X0_train), len(X1_train))
    X0_train = X0_train[:m]
    X1_train = X1_train[:m]
    X_train = np.vstack([X0_train, X1_train])
    y_train = np.array([0] * len(X0_train) + [1] * len(X1_train))

    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    Xtr = (X_train - mean) / std

    prototypes = choose_prototypes(Xtr, n_proto=min(20, len(Xtr)))
    gamma = estimate_rbf_gamma(Xtr)
    R_train = rbf_features(Xtr, prototypes, gamma)
    w = fit_logistic_regression(R_train, y_train, lr=0.15, n_steps=3500, reg=1e-4)
    return {"mean": mean, "std": std, "prototypes": prototypes, "gamma": gamma, "w": w}

def boundary_scores(model, X0_test, X1_test):
    m = min(len(X0_test), len(X1_test))
    X0_test = X0_test[:m]
    X1_test = X1_test[:m]
    X_test = np.vstack([X0_test, X1_test])
    y_test = np.array([0] * len(X0_test) + [1] * len(X1_test))
    Xte = (X_test - model["mean"]) / model["std"]
    R_test = rbf_features(Xte, model["prototypes"], model["gamma"])
    scores = decision_function_logistic(R_test, model["w"])
    probs = predict_proba_logistic(R_test, model["w"])
    return y_test, scores, probs, X_test

def evaluate_rbf_boundary(model, X0_test, X1_test):
    y_true, scores, probs, _ = boundary_scores(model, X0_test, X1_test)
    return evaluate_scores(y_true, scores, probs)

def fit_residual_regressor(context_X, residual_target, lr=0.05, n_steps=2000, reg=1e-4):
    Xb = np.column_stack([np.ones(len(context_X)), context_X])
    beta = np.zeros(Xb.shape[1])
    for _ in range(n_steps):
        pred = Xb @ beta
        err = pred - residual_target
        grad = Xb.T @ err / len(Xb)
        grad[1:] += reg * beta[1:]
        beta -= lr * grad
    return beta

def predict_residual(context_X, beta):
    Xb = np.column_stack([np.ones(len(context_X)), context_X])
    return Xb @ beta

def eval_from_scores(y_true, scores):
    probs = sigmoid(scores)
    return evaluate_scores(y_true, scores, probs)

def find_best_lambda(y_true, unified_scores, residual_pred, lambda_grid):
    best_lambda = float(lambda_grid[0])
    best_eval = None
    best_overlap = -np.inf
    for lam in lambda_grid:
        scores = unified_scores + lam * residual_pred
        cur_eval = eval_from_scores(y_true, scores)
        if cur_eval["overlap"] > best_overlap:
            best_overlap = cur_eval["overlap"]
            best_eval = cur_eval
            best_lambda = float(lam)
    return best_lambda, best_eval

def evaluate_all_models(X0_low, X1_low, X0_high, X1_high, lambda_grid=None):
    if lambda_grid is None:
        lambda_grid = np.linspace(0.0, 1.5, 16)

    m = min(len(X0_low), len(X1_low), len(X0_high), len(X1_high))
    X0_low = X0_low[:m]
    X1_low = X1_low[:m]
    X0_high = X0_high[:m]
    X1_high = X1_high[:m]

    unified = fit_rbf_boundary(np.vstack([X0_low, X0_high]), np.vstack([X1_low, X1_high]))
    spec_low = fit_rbf_boundary(X0_low, X1_low)
    spec_high = fit_rbf_boundary(X0_high, X1_high)

    def pack_eval(X0, X1, specialist_model):
        y_true, s_u, p_u, X_all = boundary_scores(unified, X0, X1)
        _, s_s, p_s, _ = boundary_scores(specialist_model, X0, X1)

        unified_eval = evaluate_scores(y_true, s_u, p_u)
        specialist_eval = evaluate_scores(y_true, s_s, p_s)

        fixed_scores = s_u + 0.5 * (s_s - s_u)
        fixed_eval = eval_from_scores(y_true, fixed_scores)

        context_X = make_contextual_features(X_all)
        residual_target = s_s - s_u
        beta = fit_residual_regressor(context_X, residual_target)
        residual_pred = predict_residual(context_X, beta)

        residual_scores = s_u + residual_pred
        residual_eval = eval_from_scores(y_true, residual_scores)

        best_lambda, weighted_eval = find_best_lambda(y_true, s_u, residual_pred, lambda_grid)

        return {
            "unified": unified_eval,
            "specialist": specialist_eval,
            "fixed": fixed_eval,
            "residual": residual_eval,
            "weighted": weighted_eval,
            "residual_mag": float(np.mean(np.abs(residual_pred))),
            "best_lambda": float(best_lambda),
        }

    low = pack_eval(X0_low, X1_low, spec_low)
    high = pack_eval(X0_high, X1_high, spec_high)

    return {
        "unified_low": low["unified"],
        "unified_high": high["unified"],
        "unified_mixed": {"accuracy": float(np.mean([low["unified"]["accuracy"], high["unified"]["accuracy"]])),
                          "overlap": float(np.mean([low["unified"]["overlap"], high["unified"]["overlap"]]))},
        "specialist_low": low["specialist"],
        "specialist_high": high["specialist"],
        "specialist_mixed": {"accuracy": float(np.mean([low["specialist"]["accuracy"], high["specialist"]["accuracy"]])),
                             "overlap": float(np.mean([low["specialist"]["overlap"], high["specialist"]["overlap"]]))},
        "fixed_low": low["fixed"],
        "fixed_high": high["fixed"],
        "fixed_mixed": {"accuracy": float(np.mean([low["fixed"]["accuracy"], high["fixed"]["accuracy"]])),
                        "overlap": float(np.mean([low["fixed"]["overlap"], high["fixed"]["overlap"]]))},
        "residual_low": low["residual"],
        "residual_high": high["residual"],
        "residual_mixed": {"accuracy": float(np.mean([low["residual"]["accuracy"], high["residual"]["accuracy"]])),
                           "overlap": float(np.mean([low["residual"]["overlap"], high["residual"]["overlap"]]))},
        "weighted_low": low["weighted"],
        "weighted_high": high["weighted"],
        "weighted_mixed": {"accuracy": float(np.mean([low["weighted"]["accuracy"], high["weighted"]["accuracy"]])),
                           "overlap": float(np.mean([low["weighted"]["overlap"], high["weighted"]["overlap"]]))},
        "residual_mag_low": low["residual_mag"],
        "residual_mag_high": high["residual_mag"],
        "best_lambda_low": low["best_lambda"],
        "best_lambda_high": high["best_lambda"],
    }

## Bootstrap helper

In [ ]:
def bootstrap_rows(X, rng):
    idx = rng.integers(0, len(X), size=len(X))
    return X[idx]

def summarize(vals):
    vals = np.array(vals)
    return {
        "mean": float(vals.mean()),
        "q025": float(np.quantile(vals, 0.025)),
        "q975": float(np.quantile(vals, 0.975)),
    }

def bootstrap_scale_weighted_residual(X0_low, X1_low, X0_high, X1_high, lambda_grid=None, n_boot=16, seed=9423):
    boot_rng = np.random.default_rng(seed)
    keys = [
        "unified_low","unified_high","unified_mixed",
        "specialist_low","specialist_high","specialist_mixed",
        "fixed_low","fixed_high","fixed_mixed",
        "residual_low","residual_high","residual_mixed",
        "weighted_low","weighted_high","weighted_mixed",
    ]
    store = {k: [] for k in keys}
    residual_mag_low = []
    residual_mag_high = []
    lambda_low = []
    lambda_high = []

    m = min(len(X0_low), len(X1_low), len(X0_high), len(X1_high))
    X0_low = X0_low[:m]
    X1_low = X1_low[:m]
    X0_high = X0_high[:m]
    X1_high = X1_high[:m]

    for _ in range(n_boot):
        A0 = bootstrap_rows(X0_low, boot_rng)
        A1 = bootstrap_rows(X1_low, boot_rng)
        B0 = bootstrap_rows(X0_high, boot_rng)
        B1 = bootstrap_rows(X1_high, boot_rng)

        out = evaluate_all_models(A0, A1, B0, B1, lambda_grid=lambda_grid)
        for key in keys:
            store[key].append(out[key]["overlap"])
        residual_mag_low.append(out["residual_mag_low"])
        residual_mag_high.append(out["residual_mag_high"])
        lambda_low.append(out["best_lambda_low"])
        lambda_high.append(out["best_lambda_high"])

    result = {key: summarize(vals) for key, vals in store.items()}
    result["residual_mag_low"] = summarize(residual_mag_low)
    result["residual_mag_high"] = summarize(residual_mag_high)
    result["best_lambda_low"] = summarize(lambda_low)
    result["best_lambda_high"] = summarize(lambda_high)
    return result

## Reduced experiment grid

In [ ]:
window_sizes = [3, 5, 7]
feature_family = "minimal"
sample_size = 110
height_block = (0, 400)
n_boot = 16
lambda_grid = np.linspace(0.0, 1.5, 16)

systems = {
    "entropy": ("low_entropy", "high_entropy"),
    "unevenness": ("low_unevenness", "high_unevenness"),
}

## Base gap slices

In [ ]:
start, stop = height_block
zeta_base_gaps = zeta_gaps_full[start:stop]
poisson_base_gaps = poisson_gaps_full[start:stop]
gue_base_gaps = gue_gaps_full[:max(stop - start + 300, 900)]

len(zeta_base_gaps), len(poisson_base_gaps), len(gue_base_gaps)

## Build condition-specific feature sets

In [ ]:
def get_conditioned_features(gaps, condition_name, k=5, feature_family="minimal", n=110):
    _, stats, X = build_windows_and_features(gaps, feature_family=feature_family, k=k)
    mask = apply_condition_mask(stats, condition_name)
    Xc = X[mask]
    return Xc[:min(len(Xc), n)]

conditioned = {}

for k in window_sizes:
    conditioned[k] = {}
    for cond_lo, cond_hi in systems.values():
        for cond in [cond_lo, cond_hi]:
            conditioned[k][("zeta", cond)] = get_conditioned_features(zeta_base_gaps, cond, k=k, feature_family=feature_family, n=sample_size)
            conditioned[k][("poisson", cond)] = get_conditioned_features(poisson_base_gaps, cond, k=k, feature_family=feature_family, n=sample_size)
            conditioned[k][("gue", cond)] = get_conditioned_features(gue_base_gaps, cond, k=k, feature_family=feature_family, n=sample_size)

{k: {key: len(val) for key, val in conditioned[k].items()} for k in window_sizes}

## Main scale-weighted residual sweep

In [ ]:
swr_results = []

for system_name, (cond_lo, cond_hi) in systems.items():
    for k in window_sizes:
        z_low = conditioned[k][("zeta", cond_lo)]
        z_high = conditioned[k][("zeta", cond_hi)]
        g_low = conditioned[k][("gue", cond_lo)]
        g_high = conditioned[k][("gue", cond_hi)]
        p_low = conditioned[k][("poisson", cond_lo)]
        p_high = conditioned[k][("poisson", cond_hi)]

        m = min(len(z_low), len(z_high), len(g_low), len(g_high), len(p_low), len(p_high), sample_size)
        if m < 40:
            continue

        zg = bootstrap_scale_weighted_residual(
            z_low[:m], g_low[:m], z_high[:m], g_high[:m],
            lambda_grid=lambda_grid, n_boot=n_boot, seed=20000 + 10 * k + len(system_name)
        )
        zp = bootstrap_scale_weighted_residual(
            z_low[:m], p_low[:m], z_high[:m], p_high[:m],
            lambda_grid=lambda_grid, n_boot=n_boot, seed=21000 + 10 * k + len(system_name)
        )

        row = {"system": system_name, "k": k, "sample_used": m}
        for prefix, out in [("zg", zg), ("zp", zp)]:
            for key, val in out.items():
                row[f"{prefix}_{key}_mean"] = val["mean"]
                row[f"{prefix}_{key}_q025"] = val["q025"]
                row[f"{prefix}_{key}_q975"] = val["q975"]

        for name in [
            "unified_low","unified_high","unified_mixed",
            "specialist_low","specialist_high","specialist_mixed",
            "fixed_low","fixed_high","fixed_mixed",
            "residual_low","residual_high","residual_mixed",
            "weighted_low","weighted_high","weighted_mixed"
        ]:
            row[f"phase_gap_{name}"] = row[f"zg_{name}_mean"] - row[f"zp_{name}_mean"]

        row["best_lambda_low"] = row["zg_best_lambda_low_mean"]
        row["best_lambda_high"] = row["zg_best_lambda_high_mean"]
        row["residual_mag_low"] = row["zg_residual_mag_low_mean"]
        row["residual_mag_high"] = row["zg_residual_mag_high_mean"]

        swr_results.append(row)

len(swr_results)

## Quick diagnostic

In [ ]:
sorted(set(r["system"] for r in swr_results)), sorted(set(r["k"] for r in swr_results))

## Weighted-residual vs residual vs unified vs specialist

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    rows = [r for r in swr_results if r["system"] == system_name]
    x = np.arange(len(window_sizes))
    width = 0.07

    metrics = [
        ("phase_gap_unified_mixed", "u-mixed"),
        ("phase_gap_specialist_mixed", "s-mixed"),
        ("phase_gap_residual_mixed", "r-mixed"),
        ("phase_gap_weighted_mixed", "w-mixed"),
        ("phase_gap_weighted_low", "w-low"),
        ("phase_gap_weighted_high", "w-high"),
    ]

    for offset, (metric, label) in zip(np.linspace(-2.5, 2.5, len(metrics)), metrics):
        vals = [next(r for r in rows if r["k"] == k)[metric] for k in window_sizes]
        ax.bar(x + offset * width, vals, width, label=label)

    ax.set_xticks(x, window_sizes)
    ax.set_title(system_name)
    ax.set_xlabel("window size k")
    ax.set_ylabel("phase gap")
    ax.legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.show()

## Best residual weight by scale

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    rows = [r for r in swr_results if r["system"] == system_name]
    lam_low = [next(r for r in rows if r["k"] == k)["best_lambda_low"] for k in window_sizes]
    lam_high = [next(r for r in rows if r["k"] == k)["best_lambda_high"] for k in window_sizes]
    ax.plot(window_sizes, lam_low, marker="o", label="lambda low")
    ax.plot(window_sizes, lam_high, marker="o", label="lambda high")
    ax.set_title(system_name)
    ax.set_xlabel("window size k")
    ax.set_ylabel("best lambda")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Weighted-residual gain heatmaps over unified

In [ ]:
def build_weighted_gain_matrix(system_name):
    rows = [r for r in swr_results if r["system"] == system_name]
    return np.array([
        [next(r for r in rows if r["k"] == k)["phase_gap_weighted_low"] - next(r for r in rows if r["k"] == k)["phase_gap_unified_low"] for k in window_sizes],
        [next(r for r in rows if r["k"] == k)["phase_gap_weighted_high"] - next(r for r in rows if r["k"] == k)["phase_gap_unified_high"] for k in window_sizes],
        [next(r for r in rows if r["k"] == k)["phase_gap_weighted_mixed"] - next(r for r in rows if r["k"] == k)["phase_gap_unified_mixed"] for k in window_sizes],
    ])

fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    M = build_weighted_gain_matrix(system_name)
    im = ax.imshow(M, aspect="auto", origin="upper")
    ax.set_title(system_name)
    ax.set_xticks(np.arange(len(window_sizes)), window_sizes)
    ax.set_yticks(np.arange(3), ["low eval", "high eval", "mixed eval"])
    ax.set_xlabel("window size k")
    ax.set_ylabel("evaluation set")

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="weighted residual gain over unified")
plt.tight_layout()
plt.show()

## Overlap comparison curves with bootstrap intervals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=False)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    rows = [r for r in swr_results if r["system"] == system_name]
    series = [
        ("zg_unified_mixed", "u-mixed"),
        ("zg_residual_mixed", "r-mixed"),
        ("zg_weighted_mixed", "w-mixed"),
    ]

    for metric, label in series:
        means, lows, highs = [], [], []
        for k in window_sizes:
            row = next(r for r in rows if r["k"] == k)
            means.append(row[f"{metric}_mean"])
            low = row[f"{metric}_mean"] - row[f"{metric}_q025"]
            high = row[f"{metric}_q975"] - row[f"{metric}_mean"]
            lows.append(max(0.0, low))
            highs.append(max(0.0, high))
        ax.errorbar(window_sizes, means, yerr=np.vstack([np.array(lows), np.array(highs)]), marker="o", capsize=4, label=label)

    ax.set_title(f"{system_name}: zeta vs GUE overlap")
    ax.set_xlabel("window size k")
    ax.set_ylabel("overlap mean")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Residual-weight summaries

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    rows = [r for r in swr_results if r["system"] == system_name]
    mag_low = [next(r for r in rows if r["k"] == k)["residual_mag_low"] for k in window_sizes]
    mag_high = [next(r for r in rows if r["k"] == k)["residual_mag_high"] for k in window_sizes]
    ax.plot(window_sizes, mag_low, marker="o", label="residual mag low")
    ax.plot(window_sizes, mag_high, marker="o", label="residual mag high")
    ax.set_title(system_name)
    ax.set_xlabel("window size k")
    ax.set_ylabel("mean |residual correction|")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Compact scale-weighted summary

In [ ]:
summary_rows = []
for system_name in ["entropy", "unevenness"]:
    for k in window_sizes:
        row = next(r for r in swr_results if r["system"] == system_name and r["k"] == k)
        summary_rows.append({
            "system": system_name,
            "k": k,
            "weighted_gain_low": float(row["phase_gap_weighted_low"] - row["phase_gap_unified_low"]),
            "weighted_gain_high": float(row["phase_gap_weighted_high"] - row["phase_gap_unified_high"]),
            "weighted_gain_mixed": float(row["phase_gap_weighted_mixed"] - row["phase_gap_unified_mixed"]),
            "weighted_minus_residual_mixed": float(row["phase_gap_weighted_mixed"] - row["phase_gap_residual_mixed"]),
            "best_lambda_low": float(row["best_lambda_low"]),
            "best_lambda_high": float(row["best_lambda_high"]),
            "residual_mag_low": float(row["residual_mag_low"]),
            "residual_mag_high": float(row["residual_mag_high"]),
        })
summary_rows

## Compact summary

In [ ]:
summary = {
    "window_sizes": window_sizes,
    "feature_family": feature_family,
    "sample_size_target": sample_size,
    "height_block": f"{height_block[0]+1}-{height_block[1]}",
    "lambda_grid": list(map(float, lambda_grid)),
    "n_boot": n_boot,
    "systems": list(systems.keys()),
    "summary_rows": summary_rows,
}
summary

## Notes

- If the weighted residual model beats the unweighted residual model, then residual strength is genuinely scale-dependent.
- If best lambda decreases or stabilizes with k, that supports the idea that large-scale windows need less residual correction per unit residual magnitude.
- If best lambda differs strongly between entropy and unevenness, then the two condition systems expose different scale laws.

## Next directions

A natural next notebook could do one of these:

1. fit a smooth lambda(k) rather than selecting from a grid
2. compare different contextual residual features
3. add nonlinear residual regressors
4. test scale-weighted residuals across feature families